In [ ]:
import pymc as pm
import numpy as np
import pandas as pd
import pytensor.tensor as pt
import seaborn as sns
import matplotlib as plt
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from tqdm.notebook import tqdm
import os

In [ ]:
data_dir = r"D:\ForestFire\CBH\data"
df_train = pd.read_csv(os.path.join(data_dir, 'NFI7-임목조사표-filtered3-training.csv'), encoding='cp949')

In [ ]:
# Extract and prepare input arrays
input_array = np.array(df_train[['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'SID', 'CR']])
H = np.asarray(input_array[:, 0], dtype=np.float32) # np.array; always make a copy / np.asarray
D = np.asarray(input_array[:, 1], dtype=np.float32)
EL = np.asarray(input_array[:, 2], dtype=np.float32)
CD = np.asarray(input_array[:, 5], dtype=np.float32)
species_sid = np.asarray(input_array[:, 6], dtype=np.int32).flatten()
unique_species_sorted = np.sort(np.unique(species_sid))
species_to_index = {sid: idx for idx, sid in enumerate(unique_species_sorted)}
index_to_species = {v : k for k, v in species_to_index.items()}
species_mapper = np.vectorize(species_to_index.get)
species_idx = species_mapper(species_sid)
y_true = np.asarray(input_array[:, 7], dtype=np.float32)
n_species = len(np.unique(species_idx))

In [ ]:
# Dummy data placeholders (to be replaced with real values)
with pm.Model() as hierarchical_model:
    # Species-level parameters: 표준정규분포로 정의
    a = pm.Normal("a", mu=0.0, sigma=1.0, shape=n_species)
    b1 = pm.Normal("b1", mu=0.0, sigma=1.0, shape=n_species)

    # Global parameters: 표준정규분포로 정의
    b2 = pm.Normal("b2", mu=0.0, sigma=1.0, shape=n_species)
    b3 = pm.Normal("b3", mu=0.0, sigma=1.0, shape=n_species)
    c1 = pm.Normal("c1", mu=0.0, sigma=1.0, shape=n_species)
    d1 = pm.Normal("d1", mu=0.0, sigma=1.0, shape=n_species)

    sigma = pm.HalfNormal("sigma", sigma=1.0)   

    # Noise standard deviation (for the standard deviation of the log likelihood)
    # sigma = pm.HalfNormal("sigma", sigma=1.0)

    # Model computation
    H_log = pt.log1p(H)
    D_log = pt.log1p(D)
    size = b1[species_idx] * (H_log / D_log) + b2[species_idx] * H_log + b3[species_idx] * D_log**2
    comp = c1[species_idx] * CD
    site = d1[species_idx] * EL
    x = a[species_idx] + size + comp + site
    cr = 1 / (1 + pt.exp(-x))

    # Likelihood
    # y_obs = pm.Normal("y_obs", mu=cr, sigma=sigma, observed=y_true)

    # Sample from posterior
    trace = pm.sample(1000, tune=1000, target_accept=0.9, cores=1, progressbar=False) # tune: warm-up steps (not included to the final result), target_accept: the step size of the sampler

# Posterior prediction using mean values
# species-specific aprameters
a_est = trace.posterior["a"].mean(dim=("chain", "draw")).values # shape of "trace.posterior["a"]": (chains, draws, species)
b1_est = trace.posterior["b1"].mean(dim=("chain", "draw")).values

# Global parameters
b2_est = trace.posterior["b2"].mean(dim=("chain", "draw")).values
b3_est = trace.posterior["b3"].mean(dim=("chain", "draw")).values
c1_est = trace.posterior["c1"].mean(dim=("chain", "draw")).values
d1_est = trace.posterior["d1"].mean(dim=("chain", "draw")).values

H_log = np.log1p(H)
D_log = np.log1p(D)
cr_pred = a_est[species_idx] + b1_est[species_idx] * (H_log / D_log) + b2_est[species_idx] * H_log + b3_est[species_idx] * D_log**2 + c1_est[species_idx] * CD + d1_est[species_idx] * EL
cr_pred = 1 / (1 + np.exp(-cr_pred))

# Evaluation
metrics_list = []

for sid in tqdm(range(len(unique_species_sorted))):
    mask = (species_idx == sid) # species_idx array와 값이 동일한 순서이기에 이를 이용하여 masking을 하는 것임.
    
    y_true_sp = y_true[mask]
    y_pred_sp = cr_pred[mask]
    
    r2 = r2_score(y_true_sp, y_pred_sp)
    mae = mean_absolute_error(y_true_sp, y_pred_sp)
    rmse = root_mean_squared_error(y_true_sp, y_pred_sp)
    
    # Get parameter values by species index
    a_val = a_est[sid]
    b1_val = b1_est[sid]
    b2_val = b2_est[sid]
    b3_val = b3_est[sid]
    c1_val = c1_est[sid]
    d1_val = d1_est[sid]
    
    metrics_list.append({
        "Species ID": index_to_species[sid],
        "R² Score": round(r2, 4),
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "a": round(a_val, 4),
        "b1": round(b1_val, 4),
        "b2": round(b2_val, 4),
        "b3": round(b3_val, 4),
        "c1": round(c1_val, 4),
        "d1": round(d1_val, 4)
    })

df_species_metrics = pd.DataFrame(metrics_list)


# display dataframe
df_species_metrics


In [ ]:
sns.scatterplot(x=y_true, y=cr_pred)